# Geração de Prontuários Sintéticos

Notebook para executar a geração de prontuários sintéticos usando o código do repositório e o Google Drive.
Fluxo:
1. Montar o Google Drive
2. Clonar ou atualizar o repositório por HTTPS
3. Instalar dependências
4. Configurar a execução
5. Instanciar e executar o use case diretamente


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!if [ -d /content/tech-challenge-fase-3/.git ]; then git -C /content/tech-challenge-fase-3 pull; else git clone https://github.com/JeffersonPantoja/tech-challenge-fase3.git /content/tech-challenge-fase-3; fi


Cloning into '/content/tech-challenge-fase-3'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (437/437), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 437 (delta 254), reused 331 (delta 148), pack-reused 0 (from 0)
Receiving objects: 100% (437/437), 20.60 MiB | 19.56 MiB/s, done.
Resolving deltas: 100% (254/254), done.


In [3]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path('/content/tech-challenge-fase-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/teach-chalenge3')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

assert PROJECT_ROOT.exists(), f'Projeto não encontrado em {PROJECT_ROOT}'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print('Projeto:', PROJECT_ROOT)
print('Drive:', DRIVE_ROOT)


Projeto: /content/tech-challenge-fase-3
Drive: /content/drive/MyDrive/teach-chalenge3


In [4]:
!pip -q install --upgrade transformers accelerate peft bitsandbytes sentencepiece huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 58.7 MB/s eta 0:00:00


In [5]:
LLAMA_MODEL_PATH = 'meta-llama/Llama-3.2-1B-Instruct'
LLAMA_MAX_NEW_TOKENS = 512
SAVE_TO_DRIVE = True
RESUME = True
BATCH_SIZE = 7
NUM_BATCHES = 100
OUTPUT_NAME = 'patient_records.jsonl'
CHECKPOINT_NAME = 'patient_records.checkpoint.json'
FAILED_CHECKPOINT_NAME = 'patient_records.failed.checkpoint.json'

print('Modelo Llama:', LLAMA_MODEL_PATH)
print('Salvar no Drive:', SAVE_TO_DRIVE)


Modelo Llama: Qwen/Qwen2.5-1.5B-Instruct
Salvar no Drive: True


In [6]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Defina o Secret HF_TOKEN no Colab'
login(token=HF_TOKEN)


In [7]:
from src.application.BuildSyntheticPatientRecordsUseCase import BuildSyntheticPatientRecordsUseCase
from src.infrastructure.JsonSyntheticPatientRecordWriter import JsonSyntheticPatientRecordWriter
from src.infrastructure.JsonSyntheticPatientRecordsCheckpointStore import JsonSyntheticPatientRecordsCheckpointStore
from src.infrastructure.JsonSyntheticPatientRecordsFailedCheckpointStore import JsonSyntheticPatientRecordsFailedCheckpointStore
from src.infrastructure.LlamaSyntheticPatientRecordGenerator import LlamaSyntheticPatientRecordGenerator
from src.infrastructure.MedQaSourcesReader import MedQaSourcesReader
from src.infrastructure.QARecordCurationService import QARecordCurationService

artifact_dir = DRIVE_ROOT if SAVE_TO_DRIVE else PROJECT_ROOT / 'resources'
output_path = artifact_dir / OUTPUT_NAME
checkpoint_path = artifact_dir / CHECKPOINT_NAME
failed_checkpoint_path = artifact_dir / FAILED_CHECKPOINT_NAME

generator = LlamaSyntheticPatientRecordGenerator(
    model_path=LLAMA_MODEL_PATH,
    max_new_tokens=LLAMA_MAX_NEW_TOKENS,
)
use_case = BuildSyntheticPatientRecordsUseCase(
    reader=MedQaSourcesReader(PROJECT_ROOT / 'resources'),
    generator=generator,
    writer=JsonSyntheticPatientRecordWriter(output_path),
    curation_service=QARecordCurationService(),
    checkpoint_store=JsonSyntheticPatientRecordsCheckpointStore(checkpoint_path),
    failed_checkpoint_store=JsonSyntheticPatientRecordsFailedCheckpointStore(failed_checkpoint_path),
    resume=RESUME,
    batch_size=BATCH_SIZE,
    num_batches=NUM_BATCHES,
)

result = use_case.execute()
print(f'Registros processados: {result.records_count}')
print(f'Saída gerada em: {result.output_path}')


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Registros processados: 0
Saída gerada em: /content/drive/MyDrive/teach-chalenge3/patient_records.jsonl
